# Instacart Market Basket Analysis: Machine Learning Pipeline

This notebook demonstrates a complete end-to-end Machine Learning pipeline to predict which products a user will buy in their next order on Instacart. We use three different modeling paradigms taught in class:
1. **Matrix Factorization (NCF)** using **MindSpore** eager mode (Neural Collaborative Filtering with user and item embeddings).
2. **XGBoost Classifier** with engineered features (including the Matrix Factorization score fed as a feature).
3. **Sequential Transformer Encoder** using **MindSpore** to model order-level chronological sequences.

### Environment Setup
- Frameworks: **MindSpore 2.9.0** (CPU) & **XGBoost 3.3.0**
- Run Kernel: **Instacart ML Environment** (configured in `.venv` to avoid version conflicts)
- Hardware Target: CPU (optimized with numpy batching to prevent graph deadlocks on Windows)


## 0. Setup and Imports
Let's import the necessary libraries and verify that MindSpore is configured correctly for eager mode on CPU.


In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mindspore as ms
import mindspore.ops as ops

# Set MindSpore eager mode on CPU
ms.set_context(mode=ms.PYNATIVE_MODE, device_target="CPU")
print("MindSpore version:", ms.__version__)
# print(sys.path)
# print(sys.path)
print("Platform CPU check:")
ms.run_check()


## 1. Data Preprocessing & Downsampling

The original dataset has **32.4 million purchases** across **206,209 users**. Training deep learning models on a CPU on the full dataset is highly memory-intensive.
Here, we sample **5,000 random users** who have train orders, and extract their complete purchase history. We split them into **80% training users (4,000)** and **20% validation users (1,000)**.
This results in ~800k prior purchases, which allows fast training (less than 4 minutes total).


In [ ]:
from src import config
from src.data_preprocessing import run_preprocessing

# Run preprocessing (skips if already generated in data/processed/ folder)
run_preprocessing(force=False)

# Show data dimensions
orders = pd.read_csv(config.ORDERS_SAMPLE_PATH)
prior_prods = pd.read_csv(config.PRIOR_PRODUCTS_SAMPLE_PATH)
train_prods = pd.read_csv(config.TRAIN_PRODUCTS_SAMPLE_PATH)

# print(orders.head())
# print(prior_prods.columns)
# print(orders.head())
# print(prior_prods.columns)
print(f"Sampled orders count: {len(orders)}")
print(f"Sampled prior purchases count: {len(prior_prods)}")
print(f"Sampled train purchases count: {len(train_prods)}")


## 2. Matrix Factorization (Neural Collaborative Filtering)

We train a Collaborative Filtering model in MindSpore by mapping user IDs and product IDs to a low-dimensional embedding space ($D=64$).
We perform **negative sampling** (4 negative products per positive purchase) and optimize the model using BCE loss to predict user affinity scores for products.
The trained model predictions will be saved and passed to XGBoost as a collaborative feature.


In [ ]:
from src.matrix_factorization import train_matrix_factorization, load_mf_model_and_mappings

# Train NCF model in MindSpore (5 epochs)
train_matrix_factorization()

# Load model and mappings
mf_model, mf_mappings = load_mf_model_and_mappings()
# print(type(mf_model))
# print(mf_mappings.keys())
# print(type(mf_model))
# print(mf_mappings.keys())
print(f"MF Model load success. User vocab size: {mf_mappings['num_users']}, Product vocab size: {mf_mappings['num_products_vocab']}")


## 3. Feature Engineering & XGBoost Classifier

For each user's next order, candidate products are products they bought in prior orders. For each candidate (User, Product), we extract:
1. **User features**: total orders, average basket size, user reorder rate, average days since prior order.
2. **Product features**: total sales count, product reorder rate, average add-to-cart position.
3. **User-Product interaction features**: total times user bought this product, purchase rate, last order distance, average add-to-cart position.
4. **Collaborative Filtering feature**: the predicted affinity score from our MindSpore Matrix Factorization model.

We train XGBoost and sweep the decision threshold on validation set to find the optimal Mean F1-score.


In [ ]:
from src.xgboost_model import train_xgboost

# Train XGBoost Classifier (includes MF score loading and candidate extraction)
xgb_model, xgb_metrics = train_xgboost()
# print(xgb_metrics.keys())
# print("best f1:", xgb_metrics['best_f1'])
# print(xgb_metrics.keys())
# print("best f1:", xgb_metrics['best_f1'])


## 4. Sequential Transformer Model

We model each user's purchase history as a sequence of orders: $O_1, O_2, \dots, O_{T-1}$.
Each order $O_t$ is embedded by taking the average of its product embeddings. We add learned position embeddings and pass the sequence through a **MindSpore Transformer Encoder**.
The final user state is projected to predict the probabilities of all products in the vocabulary for the target order $O_T$.


In [ ]:
from src.transformer_model import train_transformer

# Train Transformer model in MindSpore (5 epochs)
tf_model, tf_metrics = train_transformer()
# print(tf_metrics.keys())
# print(tf_metrics.keys())


## 5. Model Evaluation and Comparison

Let's load the results of all models and compare their ROC AUC, Mean F1-score, and training speeds.
We also evaluate the Matrix Factorization model directly on the validation candidates to get a baseline.


In [ ]:
from src.train_all import evaluate_matrix_factorization_val, evaluate_transformer_candidate_val

# Direct evaluation of MF and Transformer on candidates
mf_auc, mf_f1 = evaluate_matrix_factorization_val()
tf_cand_auc, tf_cand_f1 = evaluate_transformer_candidate_val()

# print("mf auc:", mf_auc, "tf auc:", tf_cand_auc)
# print("mf auc:", mf_auc, "tf auc:", tf_cand_auc)
# Create results table
results = [
    {
        'Model': 'Matrix Factorization (NCF)',
        'ROC AUC': mf_auc,
        'Mean F1-Score': mf_f1
    },
    {
        'Model': 'XGBoost (Classifier)',
        'ROC AUC': xgb_metrics['roc_auc'],
        'Mean F1-Score': xgb_metrics['best_f1']
    },
    {
        'Model': 'Sequential Transformer (Fair)',
        'ROC AUC': tf_cand_auc,
        'Mean F1-Score': tf_cand_f1
    },
    {
        'Model': 'Sequential Transformer (Full)',
        'ROC AUC': tf_metrics['roc_auc'],
        'Mean F1-Score': tf_metrics['f1_score']
    }
]

df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))


### Visualization of Comparison
Let's plot the comparative performance of the three models.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot ROC AUC
sns.barplot(x='Model', y='ROC AUC', data=df_results, ax=axes[0], palette='Blues_r')
axes[0].set_title('Validation ROC AUC comparison')
axes[0].set_ylim(0.4, 0.9)
axes[0].set_ylabel('ROC AUC')
axes[0].set_xticklabels(df_results['Model'], rotation=15)

# Plot F1-score
sns.barplot(x='Model', y='Mean F1-Score', data=df_results, ax=axes[1], palette='Oranges_r')
axes[1].set_title('Validation Mean F1-Score comparison')
axes[1].set_ylim(0.0, 0.45)
axes[1].set_ylabel('Mean F1-Score')
axes[1].set_xticklabels(df_results['Model'], rotation=15)

plt.tight_layout()
plt.show()


### Feature Importances in XGBoost
Let's see which features were most important to our best model (XGBoost).


In [ ]:
importances = xgb_metrics['feature_importances']
df_imp = pd.DataFrame(list(importances.items()), columns=['Feature', 'Importance']).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=df_imp, palette='viridis')
plt.title('XGBoost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()
